# Recorded JUPITER runtime

Inspect software and generation settings from a completed GPU worker. These are saved measurements, not a check of the current laptop or an active allocation.

In [1]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
from IPython.display import display

root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
sys.path.insert(0, str(root / "src"))

In [2]:
from steering_transfer.artifacts import load_artifacts

load_artifacts(root, "model_answers")
path = root / "results/transfer/frozen-transfer-v2/gemma/2273125/worker-0/metadata.json"
recorded = json.loads(path.read_text())
display(
    pd.Series(
        {
            "model": recorded["model"],
            "revision": recorded["model_revision"],
            "PyTorch": recorded["torch"],
            "Transformers": recorded["transformers"],
            "dtype": recorded["protocol"]["dtype"],
            "batch_size": recorded["protocol"]["batch_size"],
            "max_new_tokens": recorded["protocol"]["max_new_tokens"],
            "explicit_do_sample": recorded["explicit_generation_overrides"][
                "do_sample"
            ],
        },
        name="Recorded worker",
    ).to_frame()
)

,Recorded worker
model,gemma
revision,96b6f1eccf38110c56df3a15bffe176da04bfd80
PyTorch,2.9.1+cu128
Transformers,4.57.6
dtype,bfloat16
batch_size,8
max_new_tokens,2048
explicit_do_sample,False


## Live GPU check

Set `RUN_RUNTIME=1` only inside an allocated GPU kernel to check that cells execute there. This cell does not allocate a node or start a server.

In [3]:
if os.environ.get("RUN_RUNTIME", "0") == "1":
    import platform
    import sys

    import torch

    print("Host:", platform.node())
    print("Architecture:", platform.machine())
    print("Python:", sys.version.split()[0])
    print("Interpreter:", sys.executable)
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    print("Visible GPUs:", torch.cuda.device_count())
    assert torch.cuda.is_available(), (
        "Select the JUPITER GPU kernel before running this cell."
    )
    print("GPU:", torch.cuda.get_device_name(0))
    x = torch.tensor([1.0, 2.0], device="cuda")
    print("GPU calculation:", (x * 2).tolist())

## Conclusion

The completed worker used BF16, batches of eight, a 2,048-token cap and explicit greedy decoding. Saved metadata establishes that run's configuration; it does not verify a currently connected notebook kernel.